# 2026-012 Natural De-Diff D35 Analysis

This notebook details flow cytometry analysis for 2026-012 De-Diff Day 35

## Initialize Environment

In [1]:
# Load Packages
library(fcexpr)
library(dplyr)
library(tidyr)
library(ggplot2)
library(ggpubr)
library(ggsci)
library(stringr)
library(jsonlite)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘jsonlite’ was built under R version 4.4.3”


In [2]:
# Set Working Directory
setwd("/home/dalbao/AlbaoRunx3Manuscript/flow/Fig02")

# Load flow processing functions
source("/home/dalbao/AlbaoRunx3Manuscript/flow/scripts/flowProcessing.R")

# Load Workspace (or cached version if it exists)
workspace <- load_workspace_cached(
                wsp_file = "2026-012-NatDeDiffD35-Analysis.wsp",
                rds_file = "2026-012-NatDeDiffD35-Analysis.rds",
                overwrite = FALSE # Set to TRUE to reprocess WSP and overwrite cached workspace
        )

# Extract Counts
counts <- workspace[["counts"]]


Attaching package: ‘rlang’


The following objects are masked from ‘package:jsonlite’:

    flatten, unbox



Attaching package: ‘rstatix’


The following object is masked from ‘package:stats’:

    filter



Attaching package: ‘purrr’


The following objects are masked from ‘package:rlang’:

    %@%, flatten, flatten_chr, flatten_dbl, flatten_int, flatten_lgl,
    flatten_raw, invoke, splice


The following object is masked from ‘package:jsonlite’:

    flatten




# Process Gating Data
Cleanup data to report only gates below CD8.

In [3]:
# Filter only samples in "Samples" FlowJoGroup
surface <- counts %>% filter(grepl("ix", FlowJoGroup))

# Replace string "mix" with "Mix" in FlowJoGroup column
surface$FlowJoGroup <- gsub("mix", "Mix", surface$FlowJoGroup)
# Add a . before M in FlowJoGroup column
surface$FlowJoGroup <- gsub("M", ".M", surface$FlowJoGroup)
# String pslit FlowJoGroup column by ".", keep both columns as [1] Tissue and [2] Mix
surface <- surface %>% separate(FlowJoGroup, into = c("Tissue", "Mix"), sep = "\\.", remove = FALSE)

# Replace _CD8 in FileName with ""
surface$FileName <- gsub("_CD8", "", surface$FileName)
surface$SampleID <- paste0(surface$Mix, "_", str_extract(surface$FileName, "\\d(?=\\.fcs)"))

head(surface)

,FileName,PopulationFullPath,Parent,Population,Count,ParentCount,FractionOfParent,xDim,yDim,eventsInside,FilePath,FlowJoGroup,Tissue,Mix,ws,SampleID
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
47,LN_Mix1_01.fcs,root,NA,root,2137487,NA,NA,NA,NA,NA,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
48,LN_Mix1_01.fcs,Lymphocytes,root,Lymphocytes,2137487,2137487,1.000000e+00,FSC-A,SSC-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
49,LN_Mix1_01.fcs,Lymphocytes/Single Cells,Lymphocytes,Lymphocytes/Single Cells,2137487,2137487,1.000000e+00,FSC-W,FSC-H,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
50,LN_Mix1_01.fcs,Lymphocytes/Single Cells/Single Cells,Lymphocytes/Single Cells,Single Cells/Single Cells,2137487,2137487,1.000000e+00,SSC-W,SSC-H,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
51,LN_Mix1_01.fcs,Lymphocytes/Single Cells/Single Cells/CD8,Lymphocytes/Single Cells/Single Cells,CD8,1982365,2137487,9.274279e-01,Comp-PE W_561-A,Comp-BUV 396-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
52,LN_Mix1_01.fcs,Lymphocytes/Single Cells/Single Cells/CD8/Transferred,Lymphocytes/Single Cells/Single Cells/CD8,Transferred,12,1982365,6.053376e-06,Comp-FITC-A,Comp-BUV496-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1


Process df:

In [4]:
# Keep only observations in surface wherin PopulationFullPath contains
# "Lymphocytes/Single Cells/Single Cells/CD8/Transferred/"
surface <- surface %>%
    filter(grepl("Lymphocytes/Single Cells/Single Cells/CD8/Transferred/", PopulationFullPath))

# Remove "Lymphocytes/Single Cells/Single Cells/CD8/Transferred/" from PopulationFullPath
surface <- surface %>%
    mutate(PopulationFullPath = gsub("Lymphocytes/Single Cells/Single Cells/CD8/Transferred/", "", PopulationFullPath))

head(surface)

,FileName,PopulationFullPath,Parent,Population,Count,ParentCount,FractionOfParent,xDim,yDim,eventsInside,FilePath,FlowJoGroup,Tissue,Mix,ws,SampleID
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
53,LN_Mix1_01.fcs,DP,Lymphocytes/Single Cells/Single Cells/CD8/Transferred,DP,6,12,0.5000000,Comp-FITC-A,Comp-BUV496-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
54,LN_Mix1_01.fcs,SP,Lymphocytes/Single Cells/Single Cells/CD8/Transferred,SP,4,12,0.3333333,Comp-FITC-A,Comp-BUV496-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
55,LN_Mix1_01.fcs,DP/CD62L,Lymphocytes/Single Cells/Single Cells/CD8/Transferred/DP,DP/CD62L,1,6,0.1666667,Comp-APC-A,Comp-mCherry-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
56,LN_Mix1_01.fcs,DP/DN,Lymphocytes/Single Cells/Single Cells/CD8/Transferred/DP,DP/DN,0,6,0.0000000,Comp-APC-A,Comp-mCherry-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
57,LN_Mix1_01.fcs,DP/DPEC,Lymphocytes/Single Cells/Single Cells/CD8/Transferred/DP,DP/DPEC,0,6,0.0000000,Comp-PE-Cy7-A,Comp-PerCP Cy5-5-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1
58,LN_Mix1_01.fcs,DP/Early,Lymphocytes/Single Cells/Single Cells/CD8/Transferred/DP,DP/Early,4,6,0.6666667,Comp-BV650-A,Comp-PerCP Cy5-5-A,1,/D:/UF%20Dropbox/SCRPS-PipkinLab/Dominic%20Albao/Experiments/2026-012%20FateMappingTest/2026-012-FateDay35/flowjo_processed/LN_Mix1_01_CD8.fcs,LN.Mix1,LN,Mix1,Analysis.wsp,Mix1_1


In [5]:
# Select relevant columns
surface <- surface %>%
    select(SampleID, Tissue, Mix, PopulationFullPath, Count, FractionOfParent) %>% 
    # Rename FractionOfParent to Percent
    rename(Percent = FractionOfParent) %>% # mutate
    mutate(Percent = Percent * 100) # Convert to percentage
    # Renme SampleID to ID
surface <- surface %>% rename(ID = SampleID)

# Preview
head(surface)

,ID,Tissue,Mix,PopulationFullPath,Count,Percent
,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>
53,Mix1_1,LN,Mix1,DP,6,50.00000
54,Mix1_1,LN,Mix1,SP,4,33.33333
55,Mix1_1,LN,Mix1,DP/CD62L,1,16.66667
56,Mix1_1,LN,Mix1,DP/DN,0,0.00000
57,Mix1_1,LN,Mix1,DP/DPEC,0,0.00000
58,Mix1_1,LN,Mix1,DP/Early,4,66.66667


Calculate total percent:

In [6]:
# Without rounding at all
total_percent <- compute_percent_of_total(surface) %>%
    add_population_derivatives() %>%
    finalize_flow_table(round_digits = NULL)

head(total_percent, n = 10)

ID,Population,Population1Deriv,Tissue,Mix,Count,Percent,PercentOfTotal
<chr>,<chr>,<chr>,<chr>,<chr>,<dbl>,<dbl>,<dbl>
Mix1_1,DP,NA,LN,Mix1,6,50.000000,50.000000
Mix1_1,DP,NA,Spleen,Mix1,29,93.548387,93.548387
Mix1_1,DP,CD62L,LN,Mix1,1,16.666667,8.333333
Mix1_1,DP,CD62L,Spleen,Mix1,1,3.448276,1.724138
Mix1_1,DP,DN,LN,Mix1,0,0.000000,0.000000
Mix1_1,DP,DN,Spleen,Mix1,0,0.000000,0.000000
Mix1_1,DP,DPEC,LN,Mix1,0,0.000000,0.000000
Mix1_1,DP,DPEC,Spleen,Mix1,5,17.241379,8.620690
Mix1_1,DP,EEC,LN,Mix1,2,33.333333,16.666667


In [7]:
interesting <- total_percent %>%
    filter(Population1Deriv %in% c("Tcf7", "TrueTcm") | is.na(Population1Deriv))

## Plot

In [8]:
# Use theme_bw, base 12, not bold, and remove legend
theme <- theme_bw(
    base_size = 14,
    base_family = "sans"
) +
    theme(
        axis.text = element_text(size = 14, color = "black"),
        axis.title = element_text(size = 14, color = "black"),
        axis.text.x = element_text(angle = 45, hjust = 0.5, vjust = 0.5, color = "black"),
        axis.text.y = element_text(size = 12, color = "black"),
        axis.title.x = element_blank(),

        # Remove facet strip background + box
        strip.background = element_rect(fill = NA, color = NA),
        strip.text = element_text(color = "black"),  # optional: keep text styling clean

        legend.position = "none",
        plot.title = element_text(size = 12),
        panel.grid = element_blank()
    )

In [9]:
comparisons <- list(
    c("SP", "DP")
)

mix_ident <- list(

    Mix1 = c(SP = "a", DP = "b"),
    Mix2 = c(SP = "b", DP ="c")

)

cols = c(a = "#1D2B53", b = "#7E2553", c = "#FF004D")

In [10]:
for(mix in c("Mix1", "Mix2")){

    # If Population1Deriv is NA, change value to "all"
    plot.data.df <- interesting %>% filter(Mix == mix)

    plot.data.df <- plot.data.df %>% 
        mutate( 
                Population1Deriv = ifelse(is.na(Population1Deriv), "all", Population1Deriv),
                facet = factor(
                    paste(Population1Deriv, Tissue),
                    levels = c( "all Spleen", "Tcf7 Spleen", "TrueTcm Spleen",
                                "all LN", "Tcf7 LN", "TrueTcm LN")),
                # Replace SP and DP in Population column with mix_ident values
                Population = factor(    Population, levels = c("SP", "DP"),
                                        labels = c(mix_ident[[mix]]["SP"], mix_ident[[mix]]["DP"])
                )
        )

    # Ensure paired samples line up correctly: plot_padj()/rstatix pair rows
    # POSITIONALLY (i-th row of group1 with i-th row of group2 as they appear
    # in the data), NOT by ID. Sorting by ID within each facet guarantees
    # both Population groups are in the same ID order, so position == animal.
    plot.data.df <- plot.data.df %>% arrange(facet, ID)

    # comparisons is defined in terms of "SP"/"DP", but Population has just been
    # relabeled to this mix's letter codes above, so translate it through
    # mix_ident too, or plot_padj() filters on labels that no longer exist
    # in the data (empty x/y vectors -> t.test "not enough 'x' observations")
    comparisons_mix <- list(unname(mix_ident[[mix]][unlist(comparisons)]))

    # # Calculate p-values and adjusted p-values for comparisons using plot_padj function
    plot.stat.df <- plot_padj(
        data = plot.data.df,
        comparisons = comparisons_mix,
        id_col = "ID",
        y_col = "PercentOfTotal",
        x_col = "Population",
        facet_col = "facet",
        # pool_sd = TRUE,
        paired = TRUE,
        step_fraction = 0.1,
        min_step = 0.5
    )
    # Save plot.stat.df to a CSV file for inspection
    write.csv(plot.stat.df, paste0("2026-012-Fig02h-", mix, ".csv"))

    # Round plot.stat.df column foldmean to one significant digit
    # Then concatenate to p.adj.signif only if not "ns", into column p.adj.fold
    plot.stat.df <- plot.stat.df %>%
        mutate(foldmeanround = signif(foldmean, digits = 2)) %>%
        mutate(p.adj.fold = ifelse(p.adj.signif == "ns", "ns", paste0(p.adj.signif, " (", foldmeanround, "x)")))

    # Only the significant comparisons get a bracket drawn (see
    # stat_pvalue_manual below), so only these need extra headroom
    plot.stat.sig <- plot.stat.df %>% filter(p.adj < 0.05)

    p <- ggplot(plot.data.df, aes(x = Population, y = PercentOfTotal, fill = Population)) +
        geom_boxplot(outlier.shape = NA) +
        # Connect each subject's SP and DP points to identify paired samples
        geom_line(aes(group = ID), color = "black", linewidth = 0.5) +
        geom_point(size = 2) +
        facet_wrap(~ facet, ncol = 3, scales = "free_y") +
        theme +
        scale_fill_manual(values = cols) +
        # Make room for the p-value bracket: with scales = "free_y", each panel
        # auto-scales to plot.data.df alone, so the bracket drawn at y.position
        # (above the tallest box, per plot_padj()'s step_fraction/min_step) would
        # get clipped at the top of the panel without this. geom_blank() is
        # invisible but still trains the y-scale, and a small multiplier adds
        # headroom for the label text sitting above the bracket line itself.
        geom_blank( data = plot.stat.sig, aes(x = group1, y = y.position),
                    inherit.aes = FALSE) +
        scale_y_continuous(expand = expansion(mult = c(0.05, 0.25))) +
        stat_pvalue_manual(
            plot.stat.sig,
            label = "p.adj.fold",   # adjusted p-value stars
            tip.length = 0.05,
            size = 4,
            inherit.aes = FALSE       # important fix
        ) + theme (  axis.text.x = element_text(angle = 0),
                    strip.text = element_blank(),
                    strip.background = element_blank())
                
    pdf(file = paste0("2026-012-Fig02h-", mix, ".pdf"), width = 4, height = 3)
    print(p)
    dev.off()
}

Warning message:
“Removed 2 rows containing non-finite outside the scale range
(`stat_boxplot()`).”
Warning message:
“Removed 1 row containing missing values or values outside the scale range
(`geom_line()`).”
Warning message:
“Removed 2 rows containing missing values or values outside the scale range
(`geom_point()`).”
